In [11]:
import glob, json, os
import numpy as np
import pandas as pd
import torch

ROOT = os.path.abspath('.')
ORIG_FPS   = 30
TARGET_FPS = 5
FSTP       = ORIG_FPS // TARGET_FPS   # 6 frames per timestep
ACTION_DIM = 23
STATE_DIM  = 7

with open(os.path.join(ROOT, 'manifest.json')) as f:
    manifest = json.load(f)

SHARD_PATHS = sorted(glob.glob(os.path.join(ROOT, 'behavior_preencoded_shard_*.pt')))
print(f'{len(SHARD_PATHS)} shards, {len(manifest["episodes"])} manifest episodes')

4 shards, 145 manifest episodes


In [12]:
rows = []
for shard_path in SHARD_PATHS:
    shard = torch.load(shard_path, map_location='cpu', weights_only=False)
    assert len(shard) == 1, f'Expected 1 episode per shard, got {len(shard)}'
    ep = shard[0]

    tokens  = ep['tokens']         # [T, token_dim]
    actions = ep['actions']        # [T, FSTP * ACTION_DIM]
    states  = ep['states']         # [T, FSTP * STATE_DIM]

    # Identify episode in manifest via sample_idx
    meta       = manifest['episodes'][ep['sample_idx']]
    raw_length = meta['raw']['length']   # frames at 30 fps
    expected_T = raw_length // FSTP      # expected timesteps at 5 fps

    T = tokens.shape[0]
    rows.append({
        'shard':            os.path.basename(shard_path),
        'sample_idx':       ep['sample_idx'],
        'task_name':        meta['task_name'],
        'episode_index':    meta['episode_index'],
        'duration_min':     round(meta['duration_minutes'], 3),
        # Length comparison
        'raw_length_30fps': raw_length,
        'expected_T_5fps':  expected_T,
        'T_tokens':         T,
        'T_actions':        actions.shape[0],
        'T_states':         states.shape[0],
        'delta_T':          T - expected_T,
        # Dimension checks
        'token_dim':        tokens.shape[1],
        'action_dim':       actions.shape[1],
        'state_dim':        states.shape[1],
        'action_dim_ok':    actions.shape[1] == FSTP * ACTION_DIM,
        'state_dim_ok':     states.shape[1] == FSTP * STATE_DIM,
    })

df = pd.DataFrame(rows)
df

,shard,sample_idx,task_name,episode_index,duration_min,raw_length_30fps,expected_T_5fps,T_tokens,T_actions,T_states,delta_T,token_dim,action_dim,state_dim,action_dim_ok,state_dim_ok
0,behavior_preencoded_shard_00000.pt,0,turning_on_radio,2150,1.536,2764,460,461,461,461,1,1408,138,42,True,True
1,behavior_preencoded_shard_00004.pt,4,turning_on_radio,760,1.535,2763,460,461,461,461,1,1408,138,42,True,True
2,behavior_preencoded_shard_00010.pt,10,picking_up_trash,12740,2.945,5301,883,884,884,884,1,1408,138,42,True,True
3,behavior_preencoded_shard_00012.pt,12,picking_up_trash,12260,2.887,5197,866,867,867,867,1,1408,138,42,True,True


In [13]:
issues = df[~df['action_dim_ok'] | ~df['state_dim_ok'] | (df['delta_T'].abs() > 1)]
if issues.empty:
    print('All shards pass length and dimension checks.')
else:
    print(f'{len(issues)} shard(s) with issues:')
    print(issues[['shard', 'task_name', 'expected_T_5fps', 'T_tokens', 'delta_T', 'action_dim_ok', 'state_dim_ok']])

All shards pass length and dimension checks.
